In [6]:
from vertex_lite.custom import load_data,load_data_v2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np

In [7]:
def filter_data(pcts=[0.05,0.25], noises=[0.25], local=False, cluster=False):
    if cluster:
        rpath = 'resultados_cluster'
    else:
        rpath = 'resultados'

    dfs = []
    for noise in noises:
        for pct in pcts:
            dfs.append(load_data(
                f'{rpath}/noise_{noise}/pct_{pct}/sim_*/cells.parquet',
                columns=['time', 'n_lados', 'neighbor_of_mutant', 'cell_id', 'alive']
            ))

    df = pd.concat(dfs, ignore_index=True)

    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df['alive'] == 1]

    if local:
        df = df[df['neighbor_of_mutant'] == True]
    else:
        df = df.drop(columns=['neighbor_of_mutant'])

    df = (
        df[df['n_lados'] >= 3]
        .groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id', 'n_lados'])
        .size()
        .reset_index(name='n_cells')
    )

    # fraction within each simulation
    df['total_cells'] = (
        df.groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id'])['n_cells']
        .transform('sum')
    )
    df['frac_cells'] = df['n_cells'] / df['total_cells']

    return df

In [8]:
def summarize(dev):
    stats = (
        dev.groupby(["time", "mesh_type", "pct_mutant",'n_lados'])["frac_cells"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    return stats


In [9]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from matplotlib.colors import TwoSlopeNorm, rgb2hex

def plot(stats, local=False, cluster=False, max_time=5000):
    sns.set_theme(
        style="white",
        context="paper",
        rc={
            "axes.edgecolor": "black",
            "axes.linewidth": 1.0,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
        }
    )
    

    local_label = "local" if local else "global"
    cluster_label = "cluster" if cluster else "uniform"
    lados_fijos = np.arange(1, 10)   # 3,4,...,10
    norm = TwoSlopeNorm(vmin=3, vcenter=6, vmax=9)
    cmap = cm.get_cmap("BrBG")
    polygon_colors = {
        n: rgb2hex(cmap(norm(n)))
        for n in lados_fijos
    }
    for pct in sorted(stats["pct_mutant"].unique()):
        for mesh in sorted(stats["mesh_type"].unique()):
            d0 = stats[
                (stats["pct_mutant"] == pct) &
                (stats["mesh_type"] == mesh)
            ].copy()

            d0 = d0[d0["time"] <= max_time]
            if d0.empty:
                continue

            fig, ax = plt.subplots(figsize=(6.5, 4))

            lados_sorted = sorted(d0["n_lados"].unique())
            
                
            for n_lados in lados_sorted:
                d = d0[d0["n_lados"] == n_lados].sort_values("time")
                if d.empty:
                    continue
                color = polygon_colors.get(n_lados, polygon_colors[6])
                x = d["time"].to_numpy()
                y = d["mean"].to_numpy()
                sem = d["sem"].fillna(0).to_numpy()

                ax.fill_between(
                    x, y - sem, y + sem,
                    color=color, alpha=0.18, linewidth=0
                )
                ax.plot(
                    x, y,
                    color=color, linewidth=2.25,
                    label=f"{n_lados} lados"
                )

            ax.set_title(
                f"Mutant percentage {pct} | noise {mesh} | {local_label} | {cluster_label}",
                fontsize=14
            )
            ax.set_xlabel("Time step", fontsize=16)
            ax.set_ylabel("Fraction of cells", fontsize=16)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            ax.tick_params(
                axis="both", which="both", direction="out",
                length=7, labelsize=18, width=1
            )
            ax.tick_params(axis='x', rotation = 45)
            ax.legend(
                title="Polygon type",
                frameon=False,
                fontsize=11,
                title_fontsize=12
            )
            ax.xaxis.set_ticks_position("bottom")
            ax.yaxis.set_ticks_position("left")
            # ax.set_xticks([0, 1000,2000,3000,4000, 5000])

            savedir=f"figures_paper_final/nlados_time/noise_{mesh}/pct_{pct}"
            os.makedirs(savedir, exist_ok=True)
            plt.tight_layout()
            fig.savefig(
                f"{savedir}/polygon_fraction_pct_{pct}_noise_{mesh}_{local_label}_{cluster_label}_{max_time}.svg",
                format="svg"
            )
            plt.close(fig)

In [10]:
for local in [True, False]:
    for cluster in [True, False]:
        df = filter_data(local=local, cluster=cluster)

        if df.empty:
            continue

        stats = summarize(df)
        plot(stats, local=local, cluster=cluster,max_time=15000 if cluster else 2000)

/tmp/ipykernel_1607329/3348382153.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("BrBG")
/tmp/ipykernel_1607329/3348382153.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("BrBG")
/tmp/ipykernel_1607329/3348382153.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("BrBG")
/tmp/ipykernel_1607329/3348382153.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.